In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm import tqdm
import time
from json import load as load_json
import re
from bs4.element import Tag

In [2]:
_Delay = 2.5

with open("config.json", "r") as f:
        headers = load_json(f)
headers

{'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
 'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7',
 'Accept-Language': 'en-US,en;q=0.9',
 'Accept-Encoding': 'gzip, deflate, br',
 'Connection': 'keep-alive',
 'Upgrade-Insecure-Requests': '1',
 'Sec-Fetch-Dest': 'document',
 'Sec-Fetch-Mode': 'navigate',
 'Sec-Fetch-Site': 'none',
 'Sec-Fetch-User': '?1',
 'Cache-Control': 'max-age=0',
 'DNT': '1'}

In [ ]:
with open("../../data/URLs/teams_url2.txt", "r") as f:
    teams_url = f.read().splitlines()
teams_url[:5]

columns = [
        "ID",
        "Name",
        "Franchise",
        "From",
        "To",
        "Years (Yrs)",
        "Playoff Appearances (Plyfs)",
        "Championships"
        # "Conference",
        # "Division",
]

dc_url = "https://www.basketball-reference.com/leagues/NBA_2024.html"

dc_columns = [
	"Name",
        "Conference",
        "Division",
]

teams_data_p1 = pd.DataFrame(columns = columns)
teams_data_p2 = pd.DataFrame(columns = dc_columns)

In [104]:
def get_data(row: Tag, data_stat: str):
        try:
                return row.find(attrs={"data-stat": data_stat}).a.text.strip()
        except:
                try:
                        return row.find(attrs={"data-stat": data_stat}).text.strip()
                except:
                        pass

def team_info(url: str, headers: dict):
        req = requests.get(url, headers)
        
        req.encoding = 'utf-8'
        soup = BeautifulSoup(req.text, "html.parser")
        result = {}
        
        while (True):
                try:
                        info_tag = soup.select("#meta > div")[1]
                        body_tag = soup.select_one(f"#{url.split('/')[4]} > tbody")
                        break
                except:
                        soup.clear()
                        
                        req = requests.get(url, headers)
                        req.encoding = 'utf-8'
                        soup = BeautifulSoup(req.text, "html.parser")
                        time.sleep(_Delay)
        
        # `Name`
        result[columns[1]] = get_data(body_tag.select_one("tr"), "team_name")
        
        # Franchise
        try:
                result[columns[2]] = info_tag.h1.select_one("span").text
        except:
                pass
        
        # From
        # To
        # Years (Yrs)
        try:
                strong_tag = info_tag.find('strong', string=lambda text: text and ('Seasons' in text))
                if strong_tag != None:
                        pt = strong_tag.find_parent('p')
                        
                        txt = pt.get_text(separator=' ', strip=True)
                        count_match = re.search(r'(\d+)\s*;', txt)
                        if (count_match is None):
                                count_match = re.search(r'(\d+)\s+NBA', txt)
                        result[columns[5]] = int(count_match.group(1)) if count_match else None
                        seasons = re.findall(r'(\d{4})-(\d{2})', txt)
                        result[columns[3]] = int(seasons[0][0]) + 1
                        result[columns[4]] = int(seasons[1][0]) + 1
        except:
                pass
        
        # Playoff Appearances
        try:
                strong_tag = info_tag.find('strong', string=lambda text: text and ('Playoff Appearances' in text))
                if strong_tag != None:
                        pt = strong_tag.find_parent('p')
                        
                        txt = pt.get_text(separator=' ', strip=True)
                        match = re.search(r'(\d+)\s*NBA', txt)
                        if match:
                                result[columns[6]] = int(match.group(1))
                        else:
                                match = re.search(r'Playoff Appearances:\s*(\d+)', txt)
                                if match:
                                        result[columns[6]] = int(match.group(1))
        except:
                pass
        
        try:
                strong_tag = info_tag.find('strong', string=lambda text: text and ('Championships' in text))
                if strong_tag != None:
                        pt = strong_tag.find_parent('p')
                        
                        txt = pt.get_text(separator=' ', strip=True)
                        match = re.search(r'(\d+)\s*NBA', txt)
                        if match:
                                result[columns[7]] = int(match.group(1))
                        else:
                                match = re.search(r'Championships:\s*(\d+)', txt)
                                if match:
                                        result[columns[7]] = int(match.group(1))
        except:
                pass
        # Championships
        
        time.sleep(_Delay)
        return result

In [105]:
teams_data = pd.DataFrame(columns = columns)
id = 0

In [106]:
for i in tqdm(range(len(teams_url))):
        result = team_info(teams_url[i], headers)
        if result == None:
                id += 1
                continue
        
        result[columns[0]] = id
        teams_data_p1.loc[id] = result
        id += 1
        
teams_data_p1

100%|██████████| 30/30 [01:37<00:00,  3.25s/it]


,ID,Name,Franchise,From,To,Years (Yrs),Playoff Appearances (Plyfs),Championships
0,0,Oklahoma City Thunder,Oklahoma City Thunder,1968,2026,59,35,2.0
1,1,Denver Nuggets,Denver Nuggets,1968,2026,50,32,1.0
2,2,Minnesota Timberwolves,Minnesota Timberwolves,1990,2026,37,14,0.0
3,3,Los Angeles Clippers,Los Angeles Clippers,1971,2026,56,19,0.0
4,4,Dallas Mavericks,Dallas Mavericks,1981,2026,46,25,1.0
5,5,Phoenix Suns,Phoenix Suns,1969,2026,58,34,0.0
6,6,New Orleans Pelicans,New Orleans Pelicans,2003,2026,24,9,0.0
7,7,Los Angeles Lakers,Los Angeles Lakers,1949,2026,78,66,17.0
8,8,Sacramento Kings,Sacramento Kings,1949,2026,78,30,1.0
9,9,Golden State Warriors,Golden State Warriors,1947,2026,80,38,7.0


In [107]:
def get_info(body: Tag) -> pd.DataFrame:
        info = body.select("tr")
        div = ""
        result = pd.DataFrame(columns = dc_columns)
        for row in info:
                row_info = {}
                if ("thead" in row.get("class", default = "")):
                        div = row.th.strong.text
                        continue
                else:
                        row_info[dc_columns[0]] = get_data(row = row, data_stat = "team_name")
                        row_info[dc_columns[2]] = div
                result.loc[len(result)] = row_info
        return result

def get_div_and_conf(url: str, headers: dict):
        req = requests.get(url, headers)
        
        req.encoding = 'utf-8'
        soup = BeautifulSoup(req.text, "html.parser")
        result = pd.DataFrame()
        
        while True:
                try:
                        Eastern_Division_body = soup.select_one("#divs_standings_E > tbody")
                        Western_Division_body = soup.select_one("#divs_standings_W > tbody")
                        break
                except:
                        soup.clear()
                        
                        req = requests.get(url, headers)
                        req.encoding = 'utf-8'
                        soup = BeautifulSoup(req.text, "html.parser")
                        time.sleep(_Delay)
                
        
        ec_df = get_info(Eastern_Division_body)
        ec_df[dc_columns[1]] = "Eastern Conference"
        wc_df = get_info(Western_Division_body)
        wc_df[dc_columns[1]] = "Western Conference"
        result = pd.concat([ec_df, wc_df], ignore_index = True)
        
        time.sleep(_Delay)
        return result

teams_data_p2 = get_div_and_conf(dc_url, headers=headers)
teams_data_p2

,Name,Conference,Division
0,Boston Celtics,Eastern Conference,Atlantic Division
1,New York Knicks,Eastern Conference,Atlantic Division
2,Philadelphia 76ers,Eastern Conference,Atlantic Division
3,Brooklyn Nets,Eastern Conference,Atlantic Division
4,Toronto Raptors,Eastern Conference,Atlantic Division
5,Milwaukee Bucks,Eastern Conference,Central Division
6,Cleveland Cavaliers,Eastern Conference,Central Division
7,Indiana Pacers,Eastern Conference,Central Division
8,Chicago Bulls,Eastern Conference,Central Division
9,Detroit Pistons,Eastern Conference,Central Division


In [ ]:
result = teams_data_p1.merge(teams_data_p2, on = columns[1])
result.to_csv("../../data/CSVs/teams_data.csv", index = False)
result

,ID,Name,Franchise,From,To,Years (Yrs),Playoff Appearances (Plyfs),Championships,Conference,Division
0,0,Oklahoma City Thunder,Oklahoma City Thunder,1968,2026,59,35,2.0,Western Conference,Northwest Division
1,1,Denver Nuggets,Denver Nuggets,1968,2026,50,32,1.0,Western Conference,Northwest Division
2,2,Minnesota Timberwolves,Minnesota Timberwolves,1990,2026,37,14,0.0,Western Conference,Northwest Division
3,3,Los Angeles Clippers,Los Angeles Clippers,1971,2026,56,19,0.0,Western Conference,Pacific Division
4,4,Dallas Mavericks,Dallas Mavericks,1981,2026,46,25,1.0,Western Conference,Southwest Division
5,5,Phoenix Suns,Phoenix Suns,1969,2026,58,34,0.0,Western Conference,Pacific Division
6,6,New Orleans Pelicans,New Orleans Pelicans,2003,2026,24,9,0.0,Western Conference,Southwest Division
7,7,Los Angeles Lakers,Los Angeles Lakers,1949,2026,78,66,17.0,Western Conference,Pacific Division
8,8,Sacramento Kings,Sacramento Kings,1949,2026,78,30,1.0,Western Conference,Pacific Division
9,9,Golden State Warriors,Golden State Warriors,1947,2026,80,38,7.0,Western Conference,Pacific Division


In [ ]:
df = pd.read_csv("../../data/CSVs/teams_data.csv")
df["Short Name"] = pd.Series([x[43:46] for x in teams_url])
df.to_csv("../../data/CSVs/teams_data.csv", index = False)